# Grid Relocation TutorialThis notebook demonstrates Springsteel's cylindrical grid relocation feature,which reprojects spectral data to a new coordinate center. This is essentialfor tracking vortices (hurricanes, mesocyclones) that move relative to thegrid origin during time-stepping simulations.**Key concepts:**- `relocate_grid(grid, (Δx, Δy))` — returns a new grid with shifted center- `relocate_grid!(grid, (Δx, Δy))` — mutates in place (for time-stepping loops)- `grid_center(grid)` — diagnostic: cumulative center shift- Boundary strategies for points outside the original domain- Multigrid (patchChain) relocation with outer environment lookup

In [ ]:
using Springsteelusing CairoMakie

## Example 1: 2D Gaussian Vortex on RL GridBuild an axisymmetric Gaussian vortex centered at the origin. The Fourierrepresentation is dominated by `k=0` (the azimuthal mean) since the fieldis axisymmetric.

In [ ]:
# Build RL grid with 20 radial cells, domain r ∈ [0, 20]gp = SpringsteelGridParameters(geometry="RL", iMin=0.0, iMax=20.0,    num_cells=20, vars=Dict("u"=>1),    BCL=Dict("u"=>CubicBSpline.R0), BCR=Dict("u"=>CubicBSpline.R0),    max_wavenumber=Dict("default"=>-1))g = createGrid(gp)# Gaussian vortex: u(r) = exp(-r²/8)pts = getGridpoints(g)for i in 1:size(pts, 1)    g.physical[i, 1, 1] = exp(-pts[i, 1]^2 / 8.0)endspectralTransform!(g)gridTransform!(g)println("Grid: $(size(g.physical, 1)) physical points, $(size(g.spectral, 1)) spectral coefficients")println("Center before relocation: ", grid_center(g))

## Example 2: Relocate the VortexShift the center by `(3.0, 2.0)` in Cartesian coordinates. After relocation,the vortex is no longer axisymmetric in the *new* coordinate system — theFourier spectrum now has power in `k=1` and higher wavenumbers.

In [ ]:
# Relocate to new centerg_shifted = relocate_grid(g, (3.0, 2.0); boundary=:azimuthal_mean)# Note: relocate_grid (non-bang) returns a new grid.# Use relocate_grid! for center tracking via grid_center().println("Shift applied: (3.0, 2.0)")# Compare spectral content: k=0 vs k≥1b_iDim = gp.b_iDimspec_orig = g.spectral[:, 1]spec_shift = g_shifted.spectral[:, 1]k0_power_orig = sum(spec_orig[1:b_iDim].^2)k1_power_orig = sum(spec_orig[b_iDim+1:end].^2)k0_power_shift = sum(spec_shift[1:b_iDim].^2)k1_power_shift = sum(spec_shift[b_iDim+1:end].^2)println("Original:  k=0 power = $(round(k0_power_orig, digits=4)), k≥1 power = $(round(k1_power_orig, sigdigits=2))")println("Shifted:   k=0 power = $(round(k0_power_shift, digits=4)), k≥1 power = $(round(k1_power_shift, sigdigits=2))")

In [ ]:
# Visualize on Cartesian gridfunction rl_to_cartesian(grid, nx=200, ny=200)    gp = grid.params    rmax = gp.iMax    x = range(-rmax, rmax, length=nx)    y = range(-rmax, rmax, length=ny)    Z = fill(NaN, nx, ny)    eval_pts = zeros(Float64, 1, 2)    for (ix, xv) in enumerate(x), (iy, yv) in enumerate(y)        r = sqrt(xv^2 + yv^2)        if r < gp.iMin + 0.01 || r > gp.iMax - 0.01; continue; end        eval_pts[1, 1] = r        eval_pts[1, 2] = atan(yv, xv)        val = evaluate_unstructured(grid, eval_pts)        Z[ix, iy] = val[1, 1]    end    return collect(x), collect(y), Zendx, y, Z_orig = rl_to_cartesian(g)_, _, Z_shift = rl_to_cartesian(g_shifted)fig = Figure(size=(1000, 400))ax1 = Axis(fig[1, 1], title="Original (centered at origin)",           xlabel="x", ylabel="y", aspect=DataAspect())hm1 = heatmap!(ax1, x, y, Z_orig, colormap=:viridis, colorrange=(0, 1))ax2 = Axis(fig[1, 2], title="Relocated (Δx=3, Δy=2)",           xlabel="x", ylabel="y", aspect=DataAspect())hm2 = heatmap!(ax2, x, y, Z_shift, colormap=:viridis, colorrange=(0, 1))Colorbar(fig[1, 3], hm2, label="u")fig

## Example 3: Boundary StrategiesWhen the center shift is large, some new-grid points map to radii beyond theoriginal domain. Springsteel offers four strategies for handling these OOB points:- `:nan` — fill with NaN (useful for diagnostics)- `:nearest` — clamp to boundary value- `:azimuthal_mean` — use only the `k=0` coefficient (smooth far-field)- `:bc_respecting` — extrapolate using radial BCs

In [ ]:
# Large shift: many points go OOBshift = (12.0, 0.0)g_nan  = relocate_grid(g, shift; boundary=:nan)g_near = relocate_grid(g, shift; boundary=:nearest)g_azm  = relocate_grid(g, shift; boundary=:azimuthal_mean)n_nan = count(isnan, g_nan.physical[:, 1, 1])println("Boundary strategy comparison (Δx=12):")println("  :nan           — $(n_nan) NaN points out of $(size(g.physical, 1))")println("  :nearest       — max=$(round(maximum(g_near.physical[:, 1, 1]), digits=4))")println("  :azimuthal_mean — max=$(round(maximum(g_azm.physical[:, 1, 1]), digits=4))")

## Example 4: Taper ZoneThe `taper_width` parameter blends the full evaluation with the azimuthal-meanvalue near the domain boundary, preventing discontinuities at the OOB transition.

In [ ]:
g_notaper = relocate_grid(g, (10.0, 0.0); boundary=:azimuthal_mean, taper_width=0)g_taper   = relocate_grid(g, (10.0, 0.0); boundary=:azimuthal_mean, taper_width=3)println("Without taper: max=$(round(maximum(g_notaper.physical[:, 1, 1]), digits=4))")println("With taper(3): max=$(round(maximum(g_taper.physical[:, 1, 1]), digits=4))")

## Example 5: In-place Relocation and Center TrackingFor time-stepping, `relocate_grid!` mutates the grid in place. The cumulativecenter shift is tracked by `grid_center()`.

In [ ]:
g_ts = createGrid(gp)pts_ts = getGridpoints(g_ts)for i in 1:size(pts_ts, 1)    g_ts.physical[i, 1, 1] = exp(-pts_ts[i, 1]^2 / 8.0)endspectralTransform!(g_ts)gridTransform!(g_ts)# Simulate 5 time steps with small center shiftsfor step in 1:5    relocate_grid!(g_ts, (0.5, 0.3); boundary=:azimuthal_mean)    cx, cy = grid_center(g_ts)    maxval = maximum(g_ts.physical[:, 1, 1])    println("Step $step: center=($(round(cx, digits=1)), $(round(cy, digits=1))), max=$(round(maxval, digits=4))")end

## Example 6: 3D RLZ Vortex RelocationThe same API works for 3D cylindrical grids (RLZ = radial spline × Fourier ×Chebyshev). The relocation operates in the horizontal (r, λ) plane whilepreserving the vertical (z) structure.

In [ ]:
gp3d = SpringsteelGridParameters(geometry="RLZ", iMin=0.0, iMax=20.0,    kMin=0.0, kMax=10.0, num_cells=15, kDim=12,    vars=Dict("u"=>1),    BCL=Dict("u"=>CubicBSpline.R0), BCR=Dict("u"=>CubicBSpline.R0),    BCB=Dict("u"=>Chebyshev.R0), BCT=Dict("u"=>Chebyshev.R0),    max_wavenumber=Dict("default"=>-1))g3d = createGrid(gp3d)pts3d = getGridpoints(g3d)for i in 1:size(pts3d, 1)    r, z = pts3d[i, 1], pts3d[i, 3]    g3d.physical[i, 1, 1] = exp(-r^2 / 8.0) * cos(π * z / 10.0)endspectralTransform!(g3d)gridTransform!(g3d)println("RLZ grid: $(size(g3d.physical, 1)) physical points")t = @elapsed g3d_shifted = relocate_grid(g3d, (3.0, 0.0); boundary=:azimuthal_mean)println("Relocation time: $(round(t*1000, digits=1)) ms")println("Max original: $(round(maximum(g3d.physical[:, 1, 1]), digits=4))")println("Max shifted:  $(round(maximum(g3d_shifted.physical[:, 1, 1]), digits=4))")

## Example 7: Multigrid Relocation with Outer EnvironmentFor patchChain grids embedded in a coarse outer environment (e.g., a hurricanevortex inside a tropical-belt Cartesian grid), relocation can draw boundarydata from the outer grid when inner-patch points go OOB.

In [ ]:
# Build coarse RR outer environment (tropical belt)outer_gp = SpringsteelGridParameters(geometry="RR",    iMin=-30.0, iMax=30.0, jMin=-30.0, jMax=30.0,    num_cells=30, vars=Dict("u"=>1),    BCL=Dict("u"=>CubicBSpline.R0), BCR=Dict("u"=>CubicBSpline.R0),    BCD=Dict("u"=>CubicBSpline.R0), BCU=Dict("u"=>CubicBSpline.R0),    max_wavenumber=Dict("default"=>0))outer = createGrid(outer_gp)pts_o = getGridpoints(outer)for i in 1:size(pts_o, 1)    x, y = pts_o[i, 1], pts_o[i, 2]    outer.physical[i, 1, 1] = exp(-(x^2 + y^2) / 100.0)endspectralTransform!(outer)gridTransform!(outer)# Build RL patchChain embedded in the outer environmentconfig = Dict{Symbol, Any}(    :topology   => :chain,    :geometry   => "RL",    :boundaries => [0.0, 10.0, 20.0],    :cells      => 10,    :vars       => Dict("u" => 1),    :BCL        => Dict("u" => NaturalBC()),    :BCR        => Dict("u" => NaturalBC()),)mg = createMultiGrid(config; embedded_in=outer)# Initialize vortex on the chainfor patch in mg.mpg.patches    pts_p = getGridpoints(patch)    for i in 1:size(pts_p, 1)        patch.physical[i, 1, 1] = exp(-pts_p[i, 1]^2 / 8.0)    end    spectralTransform!(patch)    gridTransform!(patch)endprintln("PatchChain: $(length(mg.mpg.patches)) patches, embedded in RR outer grid")println("Center before: ", grid_center(mg))# Relocate — outer environment fills OOB on outermost patchrelocate_grid!(mg, (10.0, 5.0); boundary=:azimuthal_mean)println("Center after:  ", grid_center(mg))for (i, patch) in enumerate(mg.mpg.patches)    n_nan = count(isnan, patch.physical[:, 1, 1])    println("  Patch $i: max=$(round(maximum(patch.physical[:, 1, 1]), digits=4)), NaN count=$n_nan")end

## Summary| Feature | API | Notes ||---------|-----|-------|| Solo relocation | `relocate_grid(g, (Δx,Δy))` | Returns new grid || In-place | `relocate_grid!(g, (Δx,Δy))` | For time-stepping || Center tracking | `grid_center(g)` | Cumulative shift || Boundary: NaN | `boundary=:nan` | For diagnostics || Boundary: nearest | `boundary=:nearest` | Clamp to edge || Boundary: azimuthal mean | `boundary=:azimuthal_mean` | Smooth far-field (default) || Taper zone | `taper_width=3` | Smooth OOB transition || Multigrid | `relocate_grid!(mg, ...)` | Outer environment lookup || Snap quantization | `snap=:auto` | Aligns to outer grid nodes |**Performance:** RL relocation at ~95 ms for nc=30. RLZ relocation at ~1.1 s for nc=30kDim=16 (14× faster than unstructured evaluation path).